In [2]:
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML
import os

# --- 1. חיבור לגוגל דרייב ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
    print("✅ מחובר לגוגל דרייב בהצלחה!")
except ImportError:
    BASE_DIR = os.getcwd()
    print("⚠️ לא רץ בקולאב. משתמש בתיקייה המקומית.")

# --- 2. פונקציית טעינת נתונים מהדרייב ---
def load_data(path):
    if not os.path.exists(path):
        return None
    try:
        df = pd.read_csv(path)
        # תיקון עמודת הזמן
        time_col = 'open_time' if 'open_time' in df.columns else (df.columns[0] if 'Unnamed' in df.columns[0] else None)
        if time_col:
            df.rename(columns={time_col: 'open_time'}, inplace=True)
            df['open_time'] = pd.to_datetime(df['open_time'])
            df.set_index('open_time', inplace=True)
        return df
    except Exception as e:
        print(f"שגיאה בטעינת הקובץ: {e}")
        return None

# --- 3. לוגיקת העדכון של ה-GUI ---
def update_view(model_choice, num_candles):
    file_name = 'transformer_dashboard_data.csv' if model_choice == "Transformer" else 'lstm_dashboard_data.csv'
    path = os.path.join(BASE_DIR, file_name)

    df = load_data(path)

    if df is None:
        display(HTML(f"<div style='color:red; padding:10px; border:1px solid red;'>⚠️ קובץ לא נמצא: {file_name}<br>וודא שהרצת את האימון של המודל הזה קודם.</div>"))
        return

    # חישובי מדדים (Metrics)
    last_p = df['close'].iloc[-1]
    pred_p = df['Predicted_Close'].iloc[-1]
    pct = ((pred_p - last_p) / last_p) * 100
    signal = "🟢 LONG" if pct > 0.05 else ("🔴 SHORT" if pct < -0.05 else "⚪ NEUTRAL")

    # הצגת המדדים בעיצוב HTML
    html_header = f"""
    <div style="display: flex; justify-content: space-around; background-color: #121212; padding: 20px; border-radius: 10px; color: white; font-family: sans-serif; margin-bottom: 15px;">
        <div style="text-align: center;"> <small>מחיר נוכחי</small><br><b style="font-size: 20px;">${last_p:,.2f}</b> </div>
        <div style="text-align: center;"> <small>חיזוי {model_choice}</small><br><b style="font-size: 20px; color: {'#00ff00' if pct > 0 else '#ff4444'};">${pred_p:,.2f} ({pct:+.2f}%)</b> </div>
        <div style="text-align: center;"> <small>סיגנל</small><br><b style="font-size: 20px;">{signal}</b> </div>
    </div>
    """
    display(HTML(html_header))

    # יצירת הגרף האינטראקטיבי
    df_plot = df.tail(num_candles)
    fig = go.Figure()
    fig.add_trace(go.Candlestick(x=df_plot.index, open=df_plot['open'], high=df_plot['high'], low=df_plot['low'], close=df_plot['close'], name="Market"))
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['Predicted_Close'], line=dict(color='yellow', width=2, dash='dot'), name="Prediction"))

    fig.update_layout(template="plotly_dark", height=600, xaxis_rangeslider_visible=False, margin=dict(l=10, r=10, t=30, b=10))
    fig.show()

# --- 4. הגדרת הפקדים (Widgets) ---
display(HTML("<h2 style='color: #4A90E2;'>🚀 Crypto AI Prediction Dashboard</h2>"))

model_select = widgets.Dropdown(options=["Transformer", "LSTM"], value="Transformer", description='בחר מודל:')
candles_slider = widgets.IntSlider(value=150, min=20, max=1000, step=10, description='נרות להצגה:')

# הפעלה
widgets.interact(update_view, model_choice=model_select, num_candles=candles_slider);

⚠️ לא רץ בקולאב. משתמש בתיקייה המקומית.


interactive(children=(Dropdown(description='בחר מודל:', options=('Transformer', 'LSTM'), value='Transformer'),…